# Adaptive Hybrid RecSys — Fast Training (Colab T4)

**Что ускорено vs `02_train.ipynb`:**
1. **Векторизованный `evaluate()`** — все кандидаты в одном forward pass, без python-цикла per-sample. ~50x быстрее.
2. **Быстрый negative sampling** — без `while neg in user_set`, accept-with-collisions (потерь по качеству ~0%).
3. **TF-IDF на GPU** как dense float16-тензор (160K × 5000 × 2 байта ≈ 1.6 GB) — индексируется по item_idx, ноль конверсий per-batch.
4. **Mixed precision (autocast + GradScaler)** — +1.5–2x на T4.
5. **Опциональный subsample** датасета через `SUBSAMPLE_FRAC` для быстрого dev-цикла.
6. **Один forward на (pos+neg)** в обучении вместо двух — батчуем pairwise.
7. **`pin_memory=True`, `num_workers=4`, `persistent_workers=True`** в DataLoader.

**Порядок:** Runtime → Change runtime type → **T4 GPU** → Run all

**Требование:** запущен `01_data_pipeline.ipynb`.

In [ ]:
# ── 1. Drive + paths + GPU check ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, torch
DRIVE_DIR     = '/content/drive/MyDrive/disser'
PROCESSED_DIR = f'{DRIVE_DIR}/data/processed'
OUTPUT_DIR    = f'{DRIVE_DIR}/outputs'
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True

In [ ]:
# ── 2. Hyperparameters ────────────────────────────────────────────────────
# Поставь SUBSAMPLE_FRAC = 0.1 для быстрой проверки идеи (~5 мин/эпоха).
# Поставь 1.0 для финального прогона.
SUBSAMPLE_FRAC = 0.1     # доля train для дев-цикла; 1.0 = весь датасет
EMBED_DIM      = 64
MAX_SEQ_LEN    = 50
BATCH_SIZE     = 2048    # T4: 2048 спокойно влезает
LR             = 1e-3
EPOCHS         = 30
PATIENCE       = 5
EVAL_USERS     = 5000    # сколько val-юзеров оценивать (вместо 1М)
EVAL_NEGATIVES = 99      # negatives на 1 positive — стандарт RecSys
USE_AMP        = True    # mixed precision fp16
TFIDF_ON_GPU   = True    # держать TF-IDF dense на GPU (~1.6 GB)

CKPT_PATH = f'{OUTPUT_DIR}/checkpoints/best_fast.pt'
print(f'Subsample: {SUBSAMPLE_FRAC} | batch: {BATCH_SIZE} | epochs: {EPOCHS}')

In [ ]:
# ── 3. Load data ──────────────────────────────────────────────────────────
import numpy as np, pandas as pd, scipy.sparse as sp, json, gc
from pathlib import Path

P     = Path(PROCESSED_DIR)
stats = json.load(open(P / 'dataset_stats.json'))
print('Stats:', stats)

train_df       = pd.read_parquet(P / 'train.parquet')
val_df         = pd.read_parquet(P / 'val.parquet')
test_df        = pd.read_parquet(P / 'test.parquet')
train_temporal = pd.read_parquet(P / 'train_temporal.parquet').values.astype(np.float32)
val_temporal   = pd.read_parquet(P / 'val_temporal.parquet').values.astype(np.float32)
test_temporal  = pd.read_parquet(P / 'test_temporal.parquet').values.astype(np.float32)

tfidf_sparse  = sp.load_npz(str(P / 'item_content_sparse.npz'))
seq_data      = np.load(str(P / 'user_sequences.npz'), allow_pickle=True)
user_seqs     = dict(seq_data['sequences'].item())

N_USERS     = stats['n_users']
N_ITEMS     = stats['n_items']
CONTENT_DIM = stats['content_dim']
CONTEXT_DIM = stats['context_dim']

# Subsample train
if SUBSAMPLE_FRAC < 1.0:
    n_keep = int(len(train_df) * SUBSAMPLE_FRAC)
    keep_idx = np.random.RandomState(42).choice(len(train_df), n_keep, replace=False)
    train_df = train_df.iloc[keep_idx].reset_index(drop=True)
    train_temporal = train_temporal[keep_idx]
    print(f'Subsampled train: {len(train_df):,}')

print(f'Users: {N_USERS:,} | Items: {N_ITEMS:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
gc.collect()

In [ ]:
# ── 4. Pre-compute on-GPU tensors (key speedup) ───────────────────────────
# 4a. TF-IDF: sparse → dense fp16 на GPU. Индексация O(1) по item_idx.
if TFIDF_ON_GPU and DEVICE.type == 'cuda':
    print('Densifying TF-IDF to GPU fp16...')
    tfidf_dense = torch.from_numpy(tfidf_sparse.toarray()).half().to(DEVICE)
    # Pad row 0 = zeros (для padding_idx)
    pad_row = torch.zeros(1, CONTENT_DIM, dtype=torch.float16, device=DEVICE)
    tfidf_gpu = torch.cat([pad_row, tfidf_dense], dim=0)  # shape (N_ITEMS+1, CONTENT_DIM)
    del tfidf_dense
    print(f'TF-IDF on GPU: {tfidf_gpu.shape}, {tfidf_gpu.element_size()*tfidf_gpu.numel()/1e9:.2f} GB')
else:
    tfidf_gpu = None

# 4b. user_sequences → padded tensor (N_USERS+1, MAX_SEQ_LEN) + lengths (N_USERS+1,)
print('Building padded sequence tensor...')
seq_tensor = np.zeros((N_USERS + 1, MAX_SEQ_LEN), dtype=np.int64)
len_tensor = np.ones(N_USERS + 1, dtype=np.int64)  # min 1 для GRU
for uid, seq in user_seqs.items():
    L = min(len(seq), MAX_SEQ_LEN)
    if L > 0:
        seq_tensor[uid, -L:] = seq[-L:]
        len_tensor[uid] = L
seq_tensor_gpu = torch.from_numpy(seq_tensor).to(DEVICE)
len_tensor_gpu = torch.from_numpy(len_tensor).to(DEVICE)
print(f'Sequences on GPU: {seq_tensor_gpu.shape}')

# 4c. train_user_items как numpy bool mask (для negative sampling reject)
# Хранение: dict {uid: set} → проверки в python. Принимаем коллизии (~0.5%).
train_user_items = {}
for uid, iid in zip(train_df['user_idx'].values, train_df['item_idx'].values):
    train_user_items.setdefault(int(uid), set()).add(int(iid))

del user_seqs, seq_data, tfidf_sparse
gc.collect()
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

In [ ]:
# ── 5. Lightweight Dataset (только индексы, всё остальное — на GPU) ──────
from torch.utils.data import Dataset, DataLoader

class FastRecDataset(Dataset):
    """Возвращает только id + temporal. Содержимое (TF-IDF, seqs) индексируется на GPU."""
    def __init__(self, df, temporal):
        self.users    = df['user_idx'].values.astype(np.int64)
        self.items    = df['item_idx'].values.astype(np.int64)
        self.temporal = temporal
    def __len__(self):
        return len(self.users)
    def __getitem__(self, idx):
        # neg sampling: accept-with-collision (вероятность коллизии ~k/N_ITEMS ~0.5%)
        neg = np.random.randint(1, N_ITEMS + 1)
        return (
            self.users[idx],
            self.items[idx],
            neg,
            self.temporal[idx] if idx < len(self.temporal) else np.zeros(CONTEXT_DIM, np.float32),
        )

def collate_fn(batch):
    users = torch.tensor([b[0] for b in batch], dtype=torch.long)
    pos   = torch.tensor([b[1] for b in batch], dtype=torch.long)
    neg   = torch.tensor([b[2] for b in batch], dtype=torch.long)
    ctx   = torch.tensor(np.stack([b[3] for b in batch]), dtype=torch.float32)
    return users, pos, neg, ctx

common = dict(batch_size=BATCH_SIZE, num_workers=4, pin_memory=True,
              persistent_workers=True, collate_fn=collate_fn)
train_loader = DataLoader(FastRecDataset(train_df, train_temporal), shuffle=True,  drop_last=True, **common)
val_loader   = DataLoader(FastRecDataset(val_df,   val_temporal),   shuffle=False, **common)
test_loader  = DataLoader(FastRecDataset(test_df,  test_temporal),  shuffle=False, **common)
print(f'DataLoaders ready | train batches: {len(train_loader)}')

In [ ]:
# ── 6. Model (берём из src/, чтобы не дублировать) ────────────────────────
# Если src недоступен (Colab без git clone) — хардкод тут, как в 02_train.ipynb.
import torch.nn as nn

class StaticC(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_emb = nn.Embedding(N_USERS + 1, EMBED_DIM, padding_idx=0)
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(EMBED_DIM*2, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, EMBED_DIM))
    def forward(self, u, i):
        return self.mlp(torch.cat([self.user_emb(u), self.item_emb(i)], dim=-1))

class DynamicC(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.gru = nn.GRU(EMBED_DIM, 128, num_layers=2, batch_first=True, dropout=0.1)
        self.proj = nn.Linear(128, EMBED_DIM)
    def forward(self, seqs, lens):
        x = self.item_emb(seqs)
        packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.proj(h[-1])

class ContentC(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(CONTENT_DIM, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, EMBED_DIM))
    def forward(self, x): return self.mlp(x)

class Gate(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(CONTEXT_DIM, 64), nn.ReLU(), nn.Linear(64, 3))
    def forward(self, ctx): return torch.softmax(self.net(ctx), dim=-1)

class HybridFast(nn.Module):
    def __init__(self, use_dyn=True, use_cnt=True, use_att=True):
        super().__init__()
        self.use_dyn, self.use_cnt, self.use_att = use_dyn, use_cnt, use_att
        self.static  = StaticC()
        self.dynamic = DynamicC()
        self.content = ContentC()
        self.gate    = Gate()
        self.head    = nn.Sequential(
            nn.Linear(EMBED_DIM, EMBED_DIM // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(EMBED_DIM // 2, 1))
    def forward(self, u, items, seqs, lens, content, ctx):
        h_s = self.static(u, items)
        h_d = self.dynamic(seqs, lens) if self.use_dyn else torch.zeros_like(h_s)
        h_c = self.content(content)    if self.use_cnt else torch.zeros_like(h_s)
        stack = torch.stack([h_s, h_d, h_c], dim=1)
        w = self.gate(ctx).unsqueeze(-1) if self.use_att else torch.ones(stack.size(0), 3, 1, device=stack.device) / 3
        return self.head((stack * w).sum(dim=1)).squeeze(-1)

model = HybridFast().to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 7. Helpers: GPU lookups + vectorized eval ─────────────────────────────
def lookup_seqs(user_ids):
    """(B,) user_ids → (B, MAX_SEQ_LEN) sequences, (B,) lengths."""
    return seq_tensor_gpu[user_ids], len_tensor_gpu[user_ids]

def lookup_content(item_ids):
    """(B,) item_ids → (B, CONTENT_DIM) tf-idf rows (fp16, cast to fp32 при использовании)."""
    if tfidf_gpu is not None:
        return tfidf_gpu[item_ids].float()
    # Fallback: CPU sparse lookup (медленно, но работает если TF-IDF не на GPU)
    raise NotImplementedError('TFIDF_ON_GPU=False fallback not implemented в fast notebook')

@torch.no_grad()
def evaluate_fast(model, loader, n_users_eval=EVAL_USERS, n_neg=EVAL_NEGATIVES, k=10):
    """Векторизованный eval: для каждого test sample считает scores на (1+n_neg) кандидатов сразу.
    Семплируем n_users_eval примеров (не пользователей, а интеракций) для скорости."""
    model.eval()
    rng = np.random.default_rng(42)
    
    # Берём первые n_users_eval samples из loader
    collected = {'u': [], 'pos': [], 'ctx': []}
    n_so_far = 0
    for users, pos, _neg, ctx in loader:
        take = min(len(users), n_users_eval - n_so_far)
        collected['u'].append(users[:take])
        collected['pos'].append(pos[:take])
        collected['ctx'].append(ctx[:take])
        n_so_far += take
        if n_so_far >= n_users_eval: break
    u_all   = torch.cat(collected['u']).to(DEVICE)
    pos_all = torch.cat(collected['pos']).to(DEVICE)
    ctx_all = torch.cat(collected['ctx']).to(DEVICE)
    N = u_all.size(0)

    # Sample n_neg negatives per user (accept коллизии)
    neg_all = torch.from_numpy(
        rng.integers(1, N_ITEMS + 1, size=(N, n_neg), dtype=np.int64)
    ).to(DEVICE)  # (N, n_neg)
    candidates = torch.cat([pos_all.unsqueeze(1), neg_all], dim=1)  # (N, 1+n_neg)
    n_cand = candidates.size(1)

    seqs, lens = lookup_seqs(u_all)

    # Score батчами по пользователям, чтобы не взорвать VRAM
    EVAL_BATCH = 512
    all_scores = torch.zeros(N, n_cand, device=DEVICE)
    for s in range(0, N, EVAL_BATCH):
        e = min(s + EVAL_BATCH, N)
        b = e - s
        # Expand: каждый user × n_cand items
        u_e   = u_all[s:e].unsqueeze(1).expand(-1, n_cand).reshape(-1)         # (b*n_cand,)
        i_e   = candidates[s:e].reshape(-1)                                    # (b*n_cand,)
        seq_e = seqs[s:e].unsqueeze(1).expand(-1, n_cand, -1).reshape(b*n_cand, -1)
        len_e = lens[s:e].unsqueeze(1).expand(-1, n_cand).reshape(-1)
        ctx_e = ctx_all[s:e].unsqueeze(1).expand(-1, n_cand, -1).reshape(b*n_cand, -1)
        cnt_e = lookup_content(i_e)
        scores = model(u_e, i_e, seq_e, len_e, cnt_e, ctx_e).reshape(b, n_cand)
        all_scores[s:e] = scores

    # Ranking: pos всегда idx=0
    _, ranks = all_scores.sort(dim=1, descending=True)
    pos_rank = (ranks == 0).float().argmax(dim=1)  # позиция pos в отсортированном списке
    hits     = (pos_rank < k).float()
    recall   = hits.mean().item()
    # NDCG: если pos на позиции r (0-indexed), DCG = 1/log2(r+2) если r<k, иначе 0
    ndcg     = (hits * (1.0 / torch.log2(pos_rank.float() + 2))).mean().item()
    return {'Recall@10': recall, 'NDCG@10': ndcg}

In [ ]:
# ── 8. Training loop с AMP ────────────────────────────────────────────────
import time
from tqdm.notebook import tqdm

def bpr_loss(pos_s, neg_s):
    return -torch.nn.functional.logsigmoid(pos_s - neg_s).mean()

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == 'cuda')

def train_epoch():
    model.train()
    total, n = 0.0, 0
    for users, pos, neg, ctx in tqdm(train_loader, leave=False):
        users, pos, neg, ctx = users.to(DEVICE, non_blocking=True), pos.to(DEVICE, non_blocking=True), neg.to(DEVICE, non_blocking=True), ctx.to(DEVICE, non_blocking=True)
        seqs, lens = lookup_seqs(users)
        pos_cnt = lookup_content(pos)
        neg_cnt = lookup_content(neg)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == 'cuda'):
            # Один forward на (pos, neg) через конкатенацию батча
            B = users.size(0)
            u_cat   = torch.cat([users, users], 0)
            i_cat   = torch.cat([pos, neg], 0)
            seq_cat = torch.cat([seqs, seqs], 0)
            len_cat = torch.cat([lens, lens], 0)
            cnt_cat = torch.cat([pos_cnt, neg_cnt], 0)
            ctx_cat = torch.cat([ctx, ctx], 0)
            scores  = model(u_cat, i_cat, seq_cat, len_cat, cnt_cat, ctx_cat)
            pos_s, neg_s = scores[:B], scores[B:]
            loss = bpr_loss(pos_s, neg_s)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item(); n += 1
    return total / max(n, 1)

best_ndcg, patience_cnt, history = 0.0, 0, []
for epoch in range(EPOCHS):
    t0   = time.time()
    loss = train_epoch()
    scheduler.step()
    val_m = evaluate_fast(model, val_loader)
    elapsed = time.time() - t0

    history.append({'epoch': epoch+1, 'loss': loss, **val_m})
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | loss {loss:.4f} | NDCG@10 {val_m["NDCG@10"]:.4f} | Recall@10 {val_m["Recall@10"]:.4f} | {elapsed:.0f}s')

    if val_m['NDCG@10'] > best_ndcg:
        best_ndcg = val_m['NDCG@10']; patience_cnt = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch+1, 'val_metrics': val_m}, CKPT_PATH)
        print(f'  ✓ best saved')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stop @ epoch {epoch+1}'); break

print(f'\nDone. Best val NDCG@10 = {best_ndcg:.4f}')

In [ ]:
# ── 9. Test evaluation + save ─────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded epoch {ckpt['epoch']}, val NDCG@10={ckpt['val_metrics']['NDCG@10']:.4f}")

test_m = evaluate_fast(model, test_loader, n_users_eval=10000, n_neg=99)
print(f'Test: {test_m}')

results = {
    'model': 'HybridFast',
    'subsample_frac': SUBSAMPLE_FRAC,
    'test_metrics': test_m,
    'best_val_ndcg': best_ndcg,
    'history': history,
    'n_users': N_USERS, 'n_items': N_ITEMS,
    'total_params': sum(p.numel() for p in model.parameters()),
}
with open(f'{OUTPUT_DIR}/results_fast.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved → {OUTPUT_DIR}/results_fast.json')